# 4.1 · 线性回归 / Linear Regression

> **课程定位 / Where this fits**
> 第 1 课，**Part 4 · 监督学习：回归**。
> Lesson 1, **Part 4 · Supervised Regression**.
>
> Part 0-3 把工具、统计、预处理都备齐了，从这一课起**正式建模**。线性回归是所有监督学习的起点——预测一个**连续值**(房价、销量、温度)，模型是特征的加权和。它简单、可解释、有闭式解，是几乎所有回归面试的第一题。
> Parts 0-3 prepared the tools, stats, and preprocessing; now we **model**. Linear regression is the starting point of all supervised learning — predict a **continuous value** (house price, sales, temperature) as a weighted sum of features. Simple, interpretable, with a closed-form solution — the first regression question in almost any interview.

> 📐 **符号约定 / Notation**（详见 [`NOTATION.md`](../NOTATION.md)）
> - $\mathbf{X}$ —— 特征矩阵 ($n\times d$) / feature matrix
> - $\mathbf{w}$ —— 权重(系数)向量 / weight (coefficient) vector
> - $\hat{\mathbf{y}}=\mathbf{Xw}$ —— 预测值 / predictions
> - MSE $=\frac1n\sum(\hat y_i-y_i)^2$ —— 均方误差(损失) / mean squared error (loss)
> - $\eta$ —— 学习率 / learning rate

> 💡 **面试相关 / Interview-relevant**
> - "线性回归的损失函数 / 正规方程怎么来的"（出镜率 ★★★★★）
> - "正规方程 vs 梯度下降的取舍"（★★★★★）
> - "线性回归的假设有哪些"（★★★★★）
> - "R² / RMSE / MAE 的区别"（★★★★★）
> - "系数怎么解读 / 为什么要标准化才能比大小"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解线性回归的**损失(MSE)** 与**正规方程**的推导。
   Understand the **MSE loss** and the derivation of the **normal equation**.
2. **从零**用正规方程和梯度下降两种方法求解，并对照 sklearn。
   Solve **from scratch** via normal equation and gradient descent, and match sklearn.
3. 比较两种求解法的取舍。
   Compare the trade-offs of the two solvers.
4. 正确**解读系数**（标准化后可比大小）。
   Correctly **interpret coefficients** (comparable in magnitude after scaling).
5. 用 **R²/RMSE/MAE** 评估，并用 statsmodels 做统计推断。
   Evaluate with R²/RMSE/MAE and do statistical inference with statsmodels.

## 目录 / TOC
1. [先建直觉 + 损失函数 ⭐](#1)
2. [🏠 数据 + 正规方程(从零) ⭐](#2)
3. [梯度下降(从零) ⭐](#3)
4. [对照 sklearn + 系数解读 ⭐](#4)
5. [评估：R²/RMSE/MAE ⭐](#5)
6. [统计推断：statsmodels ⭐](#6)
7. [假设 + 小结](#7)


<a id="1"></a>
## 1. 先建直觉 + 损失函数 ⭐ / Intuition & Loss

线性回归假设目标是特征的**加权和**：$\hat y = w_0 + w_1 x_1 + \dots + w_d x_d$。我们要找一组权重 $\mathbf{w}$，让预测值尽量接近真实值。
Linear regression assumes the target is a **weighted sum** of features: $\hat y = w_0 + w_1 x_1 + \dots + w_d x_d$. We seek weights $\mathbf{w}$ that make predictions as close as possible to the truth.

"尽量接近"用**均方误差(MSE)** 衡量——所有预测误差的平方的平均：
"As close as possible" is measured by the **mean squared error (MSE)** — the average of squared prediction errors:

$$J(\mathbf{w}) = \frac{1}{n}\sum_{i=1}^{n}(\hat y_i - y_i)^2 = \frac{1}{n}\|\mathbf{Xw}-\mathbf{y}\|^2$$

为什么用**平方**而不是绝对值？(1) 平方处处可导，便于求解；(2) 它对应**高斯噪声下的最大似然**(接 2.9)。MSE 是关于 $\mathbf{w}$ 的**凸函数**——只有一个最低点，所以一定能找到全局最优。
Why **squared** rather than absolute? (1) It's differentiable everywhere, easy to solve; (2) it corresponds to **maximum likelihood under Gaussian noise** (see 2.9). MSE is a **convex** function of $\mathbf{w}$ — one single minimum, so the global optimum is guaranteed.

**两种求解方式**：直接令导数为 0 得到**正规方程**（闭式解，一步到位）；或用**梯度下降**迭代逼近（大数据时更实用）。
**Two ways to solve:** set the derivative to 0 for the **normal equation** (closed form, one shot); or iterate with **gradient descent** (more practical at scale).


<a id="2"></a>
## 2. 数据 + 正规方程(从零) ⭐ / Data & Normal Equation

我们用 **California Housing**（加州房价，sklearn 内置）：8 个特征（收入、房龄、房间数、地理位置等）预测房价中位数。
We use **California Housing** (built into sklearn): 8 features (income, house age, rooms, location...) predicting median house value.

**正规方程**：令 $\nabla_\mathbf{w} J = 0$，解出 $\mathbf{w} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$。实践中**不直接求逆**（数值不稳、慢），而用 `np.linalg.solve` 解线性方程组。
**Normal equation:** setting $\nabla_\mathbf{w} J = 0$ gives $\mathbf{w} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$. In practice **don't invert directly** (unstable, slow); use `np.linalg.solve` to solve the linear system.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=4, suppress=True)

from sklearn.datasets import fetch_california_housing
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

data = fetch_california_housing(as_frame=True)
X_df, y = data.data, data.target
print(f"California Housing: {X_df.shape}, target=房价中位数 median value (单位 10万$)")
print("特征 features:", X_df.columns.tolist())

X_tr, X_te, y_tr, y_te = train_test_split(X_df.values, y.values, test_size=0.3, random_state=0)
# 标准化(只 fit train, 防泄漏 3.4) — 标准化后系数大小才可比 / scale so coefficients are comparable
scaler = StandardScaler().fit(X_tr)
Xtr_s, Xte_s = scaler.transform(X_tr), scaler.transform(X_te)
# 在最左拼一列全 1 作偏置项, 这样 w[0] 就是截距 / add a column of 1s for the intercept
Xtr_b = np.c_[np.ones(len(Xtr_s)), Xtr_s]
Xte_b = np.c_[np.ones(len(Xte_s)), Xte_s]

def fit_ols(X, y):
    # 正规方程 w=(XᵀX)⁻¹Xᵀy; 用 solve(XᵀX, Xᵀy) 解方程组而非求逆(更稳更快)
    return np.linalg.solve(X.T @ X, X.T @ y)

w = fit_ols(Xtr_b, y_tr)
print(f"\n从零(正规方程)权重 weights:")
print(f"  w0(截距 intercept) = {w[0]:.4f}")
for name, wi in zip(data.feature_names, w[1:]):
    print(f"  {name:<12} {wi:+.4f}")


<a id="3"></a>
## 3. 梯度下降(从零) ⭐ / Gradient Descent

正规方程一步到位，但当特征数 $d$ 很大时，$\mathbf{X}^\top\mathbf{X}$ 是 $d\times d$，求解约 $O(d^3)$ 会很慢。**梯度下降**改为迭代：每步沿损失的负梯度走一小步。MSE 的梯度是
The normal equation is one-shot, but when $d$ is large, $\mathbf{X}^\top\mathbf{X}$ is $d\times d$ and solving is ~$O(d^3)$, slow. **Gradient descent** iterates instead, stepping along the negative gradient. The MSE gradient is

$$\nabla_\mathbf{w} J = \frac{2}{n}\mathbf{X}^\top(\mathbf{Xw}-\mathbf{y})$$

因为 MSE 是凸的，梯度下降一定收敛到和正规方程**完全相同**的全局最优。
Since MSE is convex, gradient descent converges to **exactly the same** global optimum as the normal equation.


In [ ]:
def fit_gd(X, y, eta=0.1, n_iter=500):
    n, d = X.shape
    w = np.zeros(d)            # 权重从 0 开始
    losses = []
    for _ in range(n_iter):
        grad = (2/n) * X.T @ (X @ w - y)    # MSE 梯度 = (2/n)Xᵀ(Xw-y)
        w -= eta * grad                      # 沿负梯度走一小步(步长 eta)
        losses.append(np.mean((X @ w - y)**2))   # 记录当前 MSE 看收敛
    return w, losses

w_gd, losses = fit_gd(Xtr_b, y_tr)
print(f"梯度下降 vs 正规方程权重 最大差异: {np.abs(w_gd - w).max():.6f}")
print("→ 殊途同归(凸问题唯一最优解) same optimum")

fig, ax = plt.subplots(figsize=(7, 3))
ax.plot(losses); ax.set_xlabel("迭代 iteration"); ax.set_ylabel("MSE"); ax.set_yscale("log")
ax.set_title("梯度下降收敛(凸→稳定下降到全局最优) GD converges")
plt.tight_layout(); plt.show()


**两种解法的取舍**（面试常问）：
**Trade-offs** (often asked):

| | 正规方程 normal equation | 梯度下降 gradient descent |
|---|---|---|
| 形式 | 闭式解，一步到位 | 迭代逼近 |
| 复杂度 | $O(d^3)$（对特征数）| $O(nd)$ 每步 |
| 大 $d$ | 慢/可能不可行 | 可行 |
| 超大 $n$ | 需一次载入 | 可用 SGD/小批，流式 |
| 超参 | 无 | 学习率、迭代数 |


<a id="4"></a>
## 4. 对照 sklearn + 系数解读 ⭐ / sklearn & Coefficients

`LinearRegression` 自动处理截距，结果应和我们的从零实现一致。**系数解读**：因为我们标准化了特征，所有系数都在同一尺度，**系数的绝对值大小可以直接比较重要性**（未标准化时不能这样比）。
`LinearRegression` handles the intercept automatically and should match our from-scratch result. **Coefficient interpretation:** since we standardized, all coefficients are on the same scale, so **their magnitudes are directly comparable as importance** (you can't compare this way without standardizing).


In [ ]:
from sklearn.linear_model import LinearRegression

lr = LinearRegression().fit(Xtr_s, y_tr)     # sklearn 内部自动算截距
print(f"sklearn 截距 = {lr.intercept_:.4f}  vs 从零 w0 = {w[0]:.4f}")
print(f"系数最大差异 = {np.abs(lr.coef_ - w[1:]).max():.6f} → 完全一致\n")

# 标准化后系数大小可比 → 当作特征重要性看 / coefficients as importance (after scaling)
coef = pd.Series(lr.coef_, index=data.feature_names).sort_values(key=abs, ascending=False)
print("系数(标准化后, 可比大小) coefficients:")
print(coef.round(3).to_string())
print(f"\n解读: MedInc(收入)系数 +{coef['MedInc']:.2f} = 收入每升高 1 个标准差, 房价中位数升约 {coef['MedInc']:.2f}(10万$)")
print("Latitude/Longitude 负系数 = 越往北/西房价越低(地理效应)")


<a id="5"></a>
## 5. 评估：R²/RMSE/MAE ⭐ / Evaluation Metrics

回归的三个核心指标（详见 Part 7）：
The three core regression metrics (detailed in Part 7):
- **R²（决定系数）**：模型解释了目标方差的百分之几。1=完美，0=和"永远预测均值"一样，可为负（比均值还差）。
  **R² (coefficient of determination):** the fraction of target variance explained. 1=perfect, 0=as good as "always predict the mean", can be negative (worse than the mean).
- **RMSE（均方根误差）**：和目标同单位；因为先平方，**对大误差惩罚重**。
  **RMSE (root mean squared error):** same unit as the target; squares first, so **penalizes large errors heavily**.
- **MAE（平均绝对误差）**：和目标同单位；对所有误差**等权**，更抗异常值。
  **MAE (mean absolute error):** same unit; weights all errors **equally**, more robust to outliers.

**MAE < RMSE 几乎总成立**；两者差距越大，说明存在越多"少数大误差"。
**MAE < RMSE almost always holds**; a bigger gap signals more "few large errors".


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

y_pred = lr.predict(Xte_s)
print(f"test R²   = {r2_score(y_te, y_pred):.4f}  (解释了 {r2_score(y_te, y_pred):.0%} 的房价方差)")
print(f"test RMSE = {np.sqrt(mean_squared_error(y_te, y_pred)):.4f} (10万$)")
print(f"test MAE  = {mean_absolute_error(y_te, y_pred):.4f} (10万$)")
print("MAE < RMSE → 有一些大误差把 RMSE 拉高了(RMSE 对大误差惩罚更重)")

fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(y_te, y_pred, alpha=0.2, s=8)
ax.plot([0, 5], [0, 5], "r--", lw=2)         # 对角线: 完美预测 / perfect-prediction line
ax.set_xlabel("真实 actual"); ax.set_ylabel("预测 predicted")
ax.set_title("预测 vs 真实(越贴对角线越好)")
plt.tight_layout(); plt.show()
print("注意 actual=5 处一条竖线: target 被截顶在 5(数据本身的陷阱), 线性模型无法预测>5")


<a id="6"></a>
## 6. 统计推断：statsmodels ⭐ / Statistical Inference

sklearn 给你预测，**statsmodels 给你统计推断**——每个系数的标准误、t 值、p 值、置信区间。这把 Part 2 的假设检验用到了回归上：**p < 0.05 说明该特征的系数显著不为 0**（在控制其他特征后仍有影响）。做"解释型"分析（哪些因素显著影响房价）时这非常重要。
sklearn gives predictions; **statsmodels gives inference** — each coefficient's standard error, t-value, p-value, confidence interval. This applies Part 2's hypothesis testing to regression: **p < 0.05 means the coefficient is significantly nonzero** (still matters after controlling for other features). Crucial for "explanatory" analysis (which factors significantly drive price).


In [ ]:
import statsmodels.api as sm

X_sm = sm.add_constant(Xtr_s)     # statsmodels 要手动加常数列(截距) / add intercept column
ols = sm.OLS(y_tr, X_sm).fit()    # 普通最小二乘 / ordinary least squares
print(ols.summary().tables[1])    # 系数表: coef, std err, t, P>|t|, [95% CI]
print("\n每个系数都附带: std err(标准误 2.9) / t值 / P>|t|(假设检验 2.6) / 95% 置信区间(2.5)")
print("p < 0.05 → 该特征显著; 这就是把 Part 2 统计学用到回归推断上")


<a id="7"></a>
## 7. 假设 + 小结 / Assumptions & Summary

线性回归（尤其做**推断**时）有几个经典假设（下一课 4.2 诊断专门检验它们）：
Linear regression (especially for **inference**) has classic assumptions (next lesson 4.2 diagnoses them):
1. **线性**：目标确实是特征的线性组合。
   **Linearity:** the target really is a linear combination of features.
2. **误差独立**：残差之间不相关（时序数据常违反）。
   **Independent errors:** residuals are uncorrelated (often violated for time series).
3. **同方差**：残差方差恒定（不随预测值变化）。
   **Homoscedasticity:** residual variance is constant.
4. **误差正态**：残差近似正态（影响 p 值/置信区间的有效性）。
   **Normal errors:** residuals are roughly normal (affects validity of p-values/CIs).
5. **无多重共线性**：特征之间不高度相关（否则系数不稳）。
   **No multicollinearity:** features aren't highly correlated (else coefficients are unstable).

```
模型: ŷ=Xw; 损失=MSE(凸, =高斯噪声 MLE)
求解: 正规方程 w=(XᵀX)⁻¹Xᵀy(用 solve 不求逆) / 梯度下降(2/n)Xᵀ(Xw-y); 凸→同一全局最优
正规方程 O(d³) 适中小 d; 梯度下降 O(nd)/步 适大数据(可 SGD 流式)
系数: 标准化后大小可比=重要性; sklearn 给预测, statsmodels 给推断(p值/CI)
评估: R²(解释方差%) / RMSE(同单位, 罚大误差) / MAE(等权, 抗异常); MAE<RMSE
假设: 线性/误差独立/同方差/正态/无多重共线性(4.2 诊断)
```

### 💡 面试速查 / Interview cheat-sheet
1. **损失=MSE**(凸, 对应高斯噪声 MLE)；正规方程 $\mathbf{w}=(\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$。
   Loss = MSE (convex, Gaussian-noise MLE); normal equation as above.
2. **正规方程 O(d³) vs 梯度下降 O(nd)/步**：大数据/高维用 GD/SGD。
   Normal equation O(d³) vs GD O(nd)/step; use GD/SGD for big/high-dim data.
3. **系数标准化后才能比大小**；statsmodels 给 p 值做推断。
   Compare coefficient magnitudes only after scaling; statsmodels gives p-values.
4. **R²/RMSE/MAE**：解释方差 / 罚大误差 / 抗异常。
   R²/RMSE/MAE: variance explained / penalize large errors / robust to outliers.
5. **五大假设**：线性、误差独立、同方差、正态、无多重共线性。
   Five assumptions: linearity, independent errors, homoscedasticity, normality, no multicollinearity.

### 下一节 / Next
**4.2 回归诊断**——线性回归的假设到底满不满足? 用残差图、QQ图、VIF 等工具逐一检验，这是回归分析的"体检"。
**4.2 Regression Diagnostics** — are the assumptions actually met? Check them with residual plots, QQ plots, VIF — the "health check" of regression.
